# E.V.O. — Self-Evolving AI Agent

A complete Colab notebook with planning, tool execution, evaluation, reflection, memory, strategy learning, and adaptive strategy selection.

In [ ]:
!pip install -q groq

from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ")
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
def calculator(expression):
    try:
        allowed = "0123456789+-*/(). "
        if not expression or not all(char in allowed for char in expression):
            return "Invalid expression"
        return eval(expression, {"__builtins__": {}}, {})
    except:
        return "Error calculating expression"

def calculator_tool(expression):
    return str(calculator(expression))

In [ ]:
class Agent:

    def __init__(self):
        self.memory = []
        self.strategies = {
            "debugging": {"success": 0, "failure": 0},
            "research": {"success": 0, "failure": 0},
            "coding": {"success": 0, "failure": 0},
            "calculator": {"success": 0, "failure": 0},
            "general": {"success": 0, "failure": 0}
        }

    def remember(self, item):
        self.memory.append(item)

    def show_memory(self):
        for item in self.memory:
            print(item)

    def strategy_score(self, strategy):
        stats = self.strategies.get(strategy, {"success": 0, "failure": 0})
        total = stats["success"] + stats["failure"]
        if total == 0:
            return 0.5
        return stats["success"] / total

    def best_strategy(self):
        return max(self.strategies, key=self.strategy_score)

    def ask_llm(self, task, lessons=""):
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": """
You are the planning brain of a self-evolving AI agent.

Available tool:
- calculator: performs safe mathematical expressions

Available strategies:
- debugging
- research
- coding
- calculator
- general

Use previous lessons to improve the current decision.

Return exactly four lines:

PLAN: <short plan>
STRATEGY: <one strategy>
ACTION: <calculator or none>
INPUT: <expression or none>

Do not add anything else.
"""
                },
                {
                    "role": "user",
                    "content": f"Task: {task}\nPrevious lessons: {lessons}"
                }
            ]
        )
        return response.choices[0].message.content.strip()

    def parse_response(self, response):
        result = {
            "PLAN": "",
            "STRATEGY": "general",
            "ACTION": "none",
            "INPUT": "none"
        }

        for line in response.splitlines():
            if ":" in line:
                key, value = line.split(":", 1)
                key = key.strip().upper()
                value = value.strip()

                if key in result:
                    result[key] = value

        return result

    def execute_tool(self, action, tool_input):
        if action.lower() == "calculator":
            return calculator_tool(tool_input)
        return "No tool was required."

    def evaluate(self, task, result):
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": """
Evaluate the agent result.

Return exactly two lines:

SUCCESS: yes or no
LESSON: <short useful lesson for future tasks>

Judge whether the result actually satisfies the task.
"""
                },
                {
                    "role": "user",
                    "content": f"Task: {task}\nResult: {result}"
                }
            ]
        )
        return response.choices[0].message.content.strip()

    def parse_evaluation(self, evaluation):
        success = "no"
        lesson = evaluation

        for line in evaluation.splitlines():
            if ":" in line:
                key, value = line.split(":", 1)
                key = key.strip().upper()
                value = value.strip()

                if key == "SUCCESS":
                    success = value.lower()
                elif key == "LESSON":
                    lesson = value

        return success, lesson

    def learn(self, task, strategy, result, evaluation):
        success, lesson = self.parse_evaluation(evaluation)

        if strategy not in self.strategies:
            self.strategies[strategy] = {"success": 0, "failure": 0}

        if success == "yes":
            self.strategies[strategy]["success"] += 1
        else:
            self.strategies[strategy]["failure"] += 1

        self.remember({
            "task": task,
            "strategy": strategy,
            "result": result,
            "success": success,
            "lesson": lesson
        })

    def lessons(self, limit=5):
        recent = self.memory[-limit:]
        return [item["lesson"] for item in recent if "lesson" in item]

    def run(self, task):
        previous_lessons = self.lessons()

        if previous_lessons:
            lesson_text = " | ".join(previous_lessons)
        else:
            lesson_text = "No previous lessons."

        response = self.ask_llm(task, lesson_text)
        decision = self.parse_response(response)

        print("\n--- Agent Decision ---")
        print("Task:", task)
        print("Plan:", decision["PLAN"])
        print("Strategy:", decision["STRATEGY"])
        print("Action:", decision["ACTION"])
        print("Input:", decision["INPUT"])

        result = self.execute_tool(
            decision["ACTION"],
            decision["INPUT"]
        )

        print("Result:", result)

        evaluation = self.evaluate(task, result)

        print("\n--- Evaluation ---")
        print(evaluation)

        self.learn(
            task,
            decision["STRATEGY"],
            result,
            evaluation
        )

        print("\n--- Learning ---")
        print("Strategy score:", round(
            self.strategy_score(decision["STRATEGY"]), 2
        ))
        print("Best learned strategy:", self.best_strategy())

        return {
            "task": task,
            "decision": decision,
            "result": result,
            "evaluation": evaluation
        }

    def show_learning(self):
        print("Strategy Performance")
        for strategy, stats in self.strategies.items():
            score = self.strategy_score(strategy)
            print(
                f"{strategy}: "
                f"success={stats['success']} "
                f"failure={stats['failure']} "
                f"score={score:.2f}"
            )

    def show_lessons(self):
        print("Learned Lessons")
        for i, item in enumerate(self.memory, 1):
            print(f"{i}. {item['lesson']}")

In [ ]:
agent = Agent()

result = agent.run("Calculate 245 multiplied by 37")

In [ ]:
result = agent.run("Calculate 1250 divided by 25 and then add 50")

In [ ]:
agent.show_learning()
print()
agent.show_lessons()

In [ ]:
agent.show_memory()

In [ ]:
def run_multiple_tasks(agent, tasks):
    results = []

    for task in tasks:
        result = agent.run(task)
        results.append(result)

    return results

tasks = [
    "Calculate 500 divided by 20",
    "Calculate 72 multiplied by 15",
    "Calculate 1000 minus 375",
    "Calculate 144 divided by 12"
]

results = run_multiple_tasks(agent, tasks)

In [ ]:
agent.show_learning()

In [ ]:
print("Best strategy:", agent.best_strategy())
print("Total memories:", len(agent.memory))

## Final Agent Architecture

```text
USER TASK
    ↓
LLM PLANNER
    ↓
STRATEGY SELECTION
    ↓
TOOL DECISION
    ↓
TOOL EXECUTION
    ↓
RESULT
    ↓
LLM EVALUATOR
    ↓
SUCCESS / FAILURE
    ↓
REFLECTION / LESSON
    ↓
MEMORY
    ↓
STRATEGY PERFORMANCE
    ↓
FUTURE TASKS USE PREVIOUS LESSONS
```